In [1]:
# !pip install LunarCalendar

In [23]:
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta, date
from itertools import cycle
from PIL import Image
import re
import matplotlib.pyplot as plt
import numpy as np
import csv
import os
import pytz

animal = ["крыса", "бык", "тигр", "кролик", "дракон", "змея", "лошадь", "коза", "обезьяна", "петух", "собака", "свинья"]
stihiya = ["дерево", "дерево", "огонь", "огонь", "почва", "почва",  "металл",  "металл", "вода", "вода"]
in_yan = ["ян", "инь"]

vis_yaer = [1920, 1924, 1928, 1932, 1936, 1940, 1944, 1948, 1952, 1956, 1960, 1964, 1968, 
            1972, 1976, 1980, 1984, 1988, 1992, 1996, 2000, 2004, 2008, 2012, 2016, 2020, 
            2024, 2028, 2032, 2036, 2040, 2044, 2048, 2052]

moon_palace = dict({1: [1920, 1942, 0, 1987, 2009, 2032], 2: [0, 1943, 1965, 1988, 2010, 0], 
    3: [1921, 1944, 1966, 0, 2011, 2033], 4: [1922, 0, 1967, 1989, 2012, 2034], 
    5: [1923, 1945, 1968, 1990, 0, 2035], 6: [1924, 1946, 0, 1991, 2013, 2036], 
    7: [0, 1947, 1969, 1992, 2014, 0], 8: [1925, 1948, 1970, 0, 2015, 2037], 
    9: [1926, 0, 1971, 1993, 2016, 2038], 10: [1927, 1949, 1972, 1994, 0, 2039], 
    11: [1928, 1950, 0, 1995, 2017, 2040], 12: [0, 1951, 1973, 1996, 2018, 0], 
    13: [1929, 1952, 1974, 0, 2019, 2041], 14: [1930, 0, 1975, 1997, 2020, 2042], 
    15: [1931, 1953, 1976, 1998, 0, 2043], 16: [1932, 1954, 0, 1999, 2021, 2044], 
    17: [0, 1955, 1977, 2000, 2022, 0], 18: [1933, 1956, 1978, 0, 2023, 2045], 
    19: [1934, 0, 1979, 2001, 2024, 2046], 20: [1935, 1957, 1980, 2002, 0, 2047], 
    21: [1936, 1958, 0, 2003, 2025, 2048], 22: [0, 1959, 1981, 2004, 2026, 0], 
    23: [1937, 1960, 1982, 0, 2027, 2049], 24: [1938, 0, 1983, 2005, 2028, 2050], 
    25: [1939, 1961, 1984, 2006, 0, 2051], 26: [1940, 1962, 0, 2007, 2029, 2052], 
    27: [0, 1963, 1985, 2008, 2030, 0], 28: [1941, 1964, 1986, 0, 2031, 2053]})

sec_step = {1: 27,
            2: 2,
            3: 2,
            4: 5,
            5: 7,
            6: 10,
            7: 12,
            8: 15,
            9: 18,
            10: 20,
            11: 23,
            12: 25}


man = ["Liv.1", "Liv.4", "Liv.3", "Gb.37/Liv.3", "Liv.5/Gb.40", "Liv.2", "Liv.8", 
       "Kid.1", "Kid.7", "Kid.3", "Bl.58/Kid.3", "Kid.4/Bl.64", "Kid.2", "Kid.10", 
       "Lu.11", "Lu.8", "Lu.9", "Co.6/Lu.9", "Co.4/Lu.7", "Lu.10", "Lu.5", 
       "Ht.9/Hg.9", "Ht.4/Hg.5", "Ht.7/Hg.7", "Si.7/Ht.7/Hg.7", 
       "Ht.5/Hg.6/Si.4", "Ht.8/Hg.8", "Ht.3/ Hg.3"]

woman = ["Gb.41", "Gb.44", "Gb.34", "Gb.37/Liv.3", "Liv.5/Gb.40", "Gb.38", "Gb.43", 
       "Bl.65", "Bl.67", "Bl.40", "Bl.58/Kid.3", "Kid.4/Bl.64", "Bl.60", "Bl.66", 
       "Co.3", "Co.1", "Co.11", "Co.6/Lu.9", "Co4/Lu.7", "Co.5", "Co.2", 
       "Si.3", "Si.1", "Si.8", "Si.7/Ht.7/Hg.7", "Ht.5/Hg.6/Si.4", "Si.5", "Si.2"]

sky = {'甲': ':green[甲]',
        '乙': ':green[乙]',
        '丙': ':red[丙]',
        '丁': ':red[丁]',
        '戊': ':orange[戊]',
        '己': ':orange[己]',
        '庚': ':darkgray[庚]',
        '辛': ':darkgray[辛]',
        '壬': ':blue[壬]',
        '癸': ':blue[癸]'}

earth = {'子': ':blue[子]',
        '丑': ':orange[丑]',
        '寅': ':green[寅]',
        '卯': ':green[卯]',
        '辰': ':orange[辰]',
        '巳': ':red[巳]',
        '午': ':red[午]',
        '未': ':orange[未]',
        '申': ':darkgray[申]',
        '酉': ':darkgray[酉]',
        '戌': ':orange[戌]',
        '亥': ':blue[亥]'}


lgbf = []
for n in range(60):
# Обработка интересующей даты
    our_date =  '25/06,2025' #input('Введите дату') # 

    our_date = vis_date = re.sub('\D', '.', our_date)
    our_date = our_date.split('.')
    d = int(our_date[0])
    m = int(our_date[1])
    y = int(our_date[2])

    our_date = date(y, m, d) + timedelta(days=n)
    # print("our_date:", our_date)


    # # Вычисляем дату наступления нового года по китайскому календарю
    # import datetime
    # from lunarcalendar import Converter, Solar, Lunar, DateNotExist

    # l = Solar(year=our_date.year, month=our_date.month, day=6)

    earth_legs = pd.read_csv("data/earth_legs.csv")
    sky_hands = pd.read_csv("data/sky_hands.csv")
    planets = pd.read_csv("data/planets.csv")
    moon_palace_df = pd.read_csv("data/moon_palace.csv")

    calendar = pd.read_csv("data/calendar.csv")
    cicle = pd.read_csv("data/cicle.csv")
    calendar['date'] = pd.to_datetime(calendar['date'])
    year_v = calendar[calendar['date']==pd.to_datetime(our_date)]['years'].values[0]
    month_v = calendar[calendar['date']==pd.to_datetime(our_date)]['months'].values[0]
    day_v = calendar[calendar['date']==pd.to_datetime(our_date)]['days'].values[0]
    day = cicle[cicle["Название_calendar"] == day_v]["Название_Русский"].values[0]
    day_iero = cicle[cicle["Название_calendar"] == day_v]["Иероглиф"].values[0]
    month_iero = cicle[cicle["Название_calendar"] == month_v]["Иероглиф"].values[0]
    year_iero = cicle[cicle["Название_calendar"] == year_v]["Иероглиф"].values[0]


    from datetime import datetime
    CURRENT_TIME = CURRENT_TIME_SOLAR = datetime.now().time().strftime('%H:%M')
    current_time_solar = datetime.now().time()

    linguibafa = []
    # print(f"Текущее административное время: {CURRENT_TIME}")
    # print(f"Среднее солнечное время: {CURRENT_TIME_SOLAR}")
    a = f"{vis_date} день: {day}"
    # print(a)
    in_yan_day = cicle[cicle['Название_calendar'] == day_v]['инь_ян'].values[0]
    b = f"День: {in_yan_day.capitalize()}"
    # print(b)
    zya_zy = cicle[cicle['Название_calendar'] == day_v]['Цзя_Цзы'].values[0]

    c = f"ЦзяЦзы дня: № {cicle[cicle['Название_calendar'] == day_v]['Цзя_Цзы'].values[0]}"
    # print(c)


    # print("ФЭЙ ТЭН БА ФА")

    feitenbafa = pd.read_csv("data/feitenbafa.csv")
    for_feitenbafa = pd.read_csv("data/for_feitenbafa.csv")

    day_predictions = feitenbafa.merge(for_feitenbafa.rename(columns={"Иероглиф":day_iero[0]}))
    feitenbafa_day = day_predictions[[day_iero[0], 'Иероглиф',	'Время',	'Канал',	'Точки']]



    time_now = datetime.now()
    current_time = CURRENT_TIME_SOLAR
    # print("Текущее время:", current_time)  
    current_hour = re.search(r"(\d*)", CURRENT_TIME_SOLAR)[0]

    feitenbafa_day_disp = feitenbafa_day.iloc[:, 1:].T
    feitenbafa_day_disp.to_csv("data/feitenbafa_day_disp.csv", index=False)
    feitenbafa_day_disp = pd.read_csv("data/feitenbafa_day_disp.csv", header=1)

    for_lin_gui_ba_fa = pd.read_csv("data/for_lin_gui_ba_fa.csv")


    print('-'*40)       
    print(f"{c} // {cicle[cicle['Название_calendar'] == day_v]['Иероглиф'].values[0]}")

    linguibafa.append(cicle[cicle['Название_calendar'] == day_v]['Цзя_Цзы'].values[0])
    linguibafa.append(cicle[cicle['Название_calendar'] == day_v]['Иероглиф'].values[0])

    # print("ЛИН ГУЙ БА ФА")

    for i in feitenbafa_day.index:
        summ = sky_hands[sky_hands['Иероглиф']==day_iero[0]]['i_day'].values[0] + \
                    sky_hands[sky_hands['Иероглиф']==feitenbafa_day.iloc[i, 0]]['i_hour'].values[0] + \
                    earth_legs[earth_legs['Иероглиф']==day_iero[1]]['j_day'].values[0] + \
                    earth_legs[earth_legs['Иероглиф']==feitenbafa_day.iloc[i, 1]]['j_hour'].values[0]

        if cicle[cicle['Иероглиф']==day_iero]['инь_ян'].values[0] == 'ян':
            res = summ%9
            if res == 0:
                res = 9

        else:
            res = summ%6
            if res == 0:
                res = 6  

    # if res==5:
        linguibafa_lst = list(feitenbafa_day.iloc[i,:3].values)
        linguibafa_lst.extend(for_lin_gui_ba_fa[for_lin_gui_ba_fa['res']==res].values[0][1:])
        
        
        if 'Чжун май' in linguibafa_lst:
            print(linguibafa_lst[1], linguibafa_lst[2])
            linguibafa.append(linguibafa_lst[1])   
            linguibafa.append(linguibafa_lst[2])   

        # linguibafa_df = pd.DataFrame(
        #     data=linguibafa,
        #     columns=[feitenbafa_day.columns[0], feitenbafa_day.columns[1], feitenbafa_day.columns[2],"Канал", "Точка", "Название_точки"]
        # )


        # linguibafa_df[linguibafa_df.columns[1:]].T.to_csv("data/linguibafa_df_disp.csv", index=False)
        # linguibafa_df_disp = pd.read_csv("data/linguibafa_df_disp.csv", header=1)

    print(linguibafa)
    lgbf.append(linguibafa)
print(lgbf)

----------------------------------------
ЦзяЦзы дня: № 2 // 乙丑
子 23:00 - 01:00
申 15:00 - 17:00
亥 21:00 - 23:00
[2, '乙丑', '子', '23:00 - 01:00', '申', '15:00 - 17:00', '亥', '21:00 - 23:00']
----------------------------------------
ЦзяЦзы дня: № 3 // 丙寅
丑 01:00 - 03:00
[3, '丙寅', '丑', '01:00 - 03:00']
----------------------------------------
ЦзяЦзы дня: № 4 // 丁卯
寅 3:00 - 5:00
戌 19:00 - 21:00
[4, '丁卯', '寅', '3:00 - 5:00', '戌', '19:00 - 21:00']
----------------------------------------
ЦзяЦзы дня: № 5 // 戊辰
子 23:00 - 01:00
申 15:00 - 17:00
[5, '戊辰', '子', '23:00 - 01:00', '申', '15:00 - 17:00']
----------------------------------------
ЦзяЦзы дня: № 6 // 己巳
子 23:00 - 01:00
卯 5:00 - 7:00
亥 21:00 - 23:00
[6, '己巳', '子', '23:00 - 01:00', '卯', '5:00 - 7:00', '亥', '21:00 - 23:00']
----------------------------------------
ЦзяЦзы дня: № 7 // 庚午
子 23:00 - 01:00
申 15:00 - 17:00
[7, '庚午', '子', '23:00 - 01:00', '申', '15:00 - 17:00']
----------------------------------------
ЦзяЦзы дня: № 8 // 辛未
午 11:00 - 13:

In [24]:
lgbf_df = pd.DataFrame(
    data=lgbf,
    columns=['№','Цзя Цзы', 'Стража', 'Время', 'Стража', 'Время', 'Стража', 'Время']
)

In [26]:
lgbf_df.fillna(' ', inplace=True)

In [27]:
lgbf_df

,№,Цзя Цзы,Стража,Время,Стража,Время,Стража,Время
0,2,乙丑,子,23:00 - 01:00,申,15:00 - 17:00,亥,21:00 - 23:00
1,3,丙寅,丑,01:00 - 03:00,,,,
2,4,丁卯,寅,3:00 - 5:00,戌,19:00 - 21:00,,
3,5,戊辰,子,23:00 - 01:00,申,15:00 - 17:00,,
4,6,己巳,子,23:00 - 01:00,卯,5:00 - 7:00,亥,21:00 - 23:00
5,7,庚午,子,23:00 - 01:00,申,15:00 - 17:00,,
6,8,辛未,午,11:00 - 13:00,酉,17:00 - 19:00,,
7,9,壬申,丑,01:00 - 03:00,酉,17:00 - 19:00,,
8,10,癸酉,丑,01:00 - 03:00,酉,17:00 - 19:00,,
9,11,甲戌,卯,5:00 - 7:00,亥,21:00 - 23:00,,


In [6]:
feitenbafa_day

,甲,Иероглиф,Время,Канал,Точки
0,甲,子,23:00 - 01:00,Чонг-май,Sp.4 + Hg.6
1,乙,丑,01:00 - 03:00,Ян-цяо,Bl.62 + Si.3
2,丙,寅,3:00 - 5:00,Инь-вэй,Hg.6 + Sp.4
3,丁,卯,5:00 - 7:00,Инь-цяо,Kid.6 + Lu.7
4,戊,辰,7:00 - 9:00,Дай-май,Gb.41 + Th.5
5,己,巳,9:00 - 11:00,Жэнь-май,Lu.7 + Kid.6
6,庚,午,11:00 - 13:00,Ян-вэй,Th.5 + Gb.41
7,辛,未,13:00 - 15:00,Ду-май,Si.3 + Bl.62
8,壬,申,15:00 - 17:00,Чонг-май,Sp.4 + Hg.6
9,癸,酉,17:00 - 19:00,Ян-цяо,Bl.62 + Si.3


In [8]:
cicle[cicle['Название_calendar'] == day_v]

,Цзя_Цзы,инь_ян,Иероглиф,Название_Китай,Название_Русский,Название_calendar
0,1,ян,甲子,Цзя Цзы,Деревянная Крыса,Крыса ян дерево


#### ТАЙ ЯН БА ФА

In [ ]:
list_tai = os.listdir("data/tai_yan_ba_fa/")
for l in list_tai:
    if day_iero[0] in l:
        file=re.findall(f'(\w*{day_iero[0]}\w*.csv)', l)

tai_yan_ba_fa = pd.read_csv(f"data/tai_yan_ba_fa/{file[0]}")

try:
    current_hour_taiyan = tai_yan_ba_fa.iloc[tai_yan_ba_fa[tai_yan_ba_fa["0"]==current_hour_china[1]].index[0]:
                                                tai_yan_ba_fa[tai_yan_ba_fa["0"]==current_hour_china[1]].index[0] + 2]
except:
    current_hour_taiyan = tai_yan_ba_fa.iloc[tai_yan_ba_fa[tai_yan_ba_fa["0"]==current_hour_china[1]].index[0]:]
# current_hour_taiyan.rename(columns=current_hour_taiyan.iloc[0]).drop(current_hour_taiyan.index[0])

for i in tai_yan_ba_fa.index:
    for j in tai_yan_ba_fa.columns:
        if i%2==0:
            tai_yan_ba_fa.iloc[i, int(j)] = tai_yan_ba_fa.iloc[i, int(j)] + " " + tai_yan_ba_fa.iloc[i+1, int(j)]
            
df = tai_yan_ba_fa.drop(tai_yan_ba_fa.index[range(1,42, 2)], axis=0).reset_index(drop=True)
df.iloc[:,0] = df.iloc[:,0].str.strip()
df.iloc[:,1] = df.iloc[:,1].str.strip()
# df = df.rename(columns=df.iloc[0]).drop(df.index[0])

for i in df.index:
    start_time = datetime(y, m, d, hour=int(df.iloc[i,1].split(" - ")[0].split(".")[0]), minute=int(df.iloc[i,1].split(" - ")[0].split(".")[1])).time()
    end_time = datetime(y, m, d, hour=int(df.iloc[i,1].split(" - ")[1].split(".")[0]), minute=int(df.iloc[i,1].split(" - ")[1].split(".")[1])).time()
    if (current_time_solar >= start_time) & (current_time_solar < end_time):
        ser = df.iloc[i, :]
        ind = i
        break

# df.style.set_properties(color="red", align="right")  # doctest: +SKIP
df = df.style.set_properties(**{'background-color': 'yellow'}, subset=ind)
# df.map(lambda x: "background-color: 'yellow'", subset=current_hour_china[1])
df

In [ ]:
current_hour_china[1]

In [ ]:
da_syao = pd.read_csv("data/da_syao.csv")

current_hour_china_list = da_syao[current_hour_china[1]].to_list()
            
            
" || ".join(current_hour_china_list[1:])

In [ ]:
preproc = pd.read_excel("data/preproc.xlsx")
preproc.fillna(0, inplace=True)
preproc[preproc.columns[:3]] = preproc[preproc.columns[:3]].astype("int32")
preproc['ЗАПРЕТЫ'] = preproc['ЗАПРЕТЫ'].str.strip()
preproc

In [ ]:
# preproc.to_csv("data/day_sky_veto.csv", index=False)

In [ ]:
earth_legs['Иероглиф'].values

In [ ]:
color_dict = {'甲':'green', '乙':'green', '丙':'red', '丁':'red', '戊':'orange', '己':'orange', '庚':'grey', '辛':'grey', '壬':'blue', '癸':'blue',
                '子':'blue', '丑':'orange', '寅':'green', '卯':'green', '辰':'orange', '巳':'red', '午':'red', '未':'orange', '申':'grey', '酉':'grey', '戌':'orange', '亥':'blue'}

# Пример DataFrame с текстовыми строками
data = {
    'Text': ['乙子', '丁未', '壬卯']
}

df = pd.DataFrame(data)

# Функция для окрашивания отдельных слов
def highlight_words(text):
    # Окрасим "Hello" в красный, а "World" в зелёный
    highlighted_text = text.replace(text[0], f'<span style="color:{color_dict[text[0]]};font-weight: bold">{text[0]}</span>')
    highlighted_text = highlighted_text.replace(text[1], f'<span style="color:{color_dict[text[1]]};font-weight: bold">{text[1]}</span>')
    return highlighted_text

# Применяем функцию к колонке 'Text' и выводим как HTML
df['Text'] = df['Text'].apply(highlight_words)

# Отображаем стилизованный DataFrame
styled_df = df.style.set_table_attributes('style="width: 50%;"')
styled_df

In [ ]:
import pandas as pd

# Создаем пример DataFrame
data = {
    'Название': ['Товар A', 'Товар B', 'Товар C'],
    'Описание': ['Это описание товара A.<br>Дополнительная информация.', 
                 'Этот товар является Б.<br>Подробности о товаре.', 
                 'Товар C.<br>Содержит много интересного.']
}

df = pd.DataFrame(data)

# Функция для стилизации текста в ячейках
def highlight_bold(s):
    return ['font-weight: bold; color: blue' for _ in s]

# Создаем стилизованный DataFrame и применяем стили к названиям столбцов
styled_df = df.style.apply(highlight_bold, subset=['Название'])
styled_df.set_table_attributes('style="width: 100%; border-collapse: collapse;"')

# Применяем стили к заголовкам столбцов
styled_df.set_table_styles({
    'Название': [{'selector': 'th', 'props': [('font-weight', 'bold'), ('color', 'blue')]}],
    'Описание': [{'selector': 'th', 'props': [('font-weight', 'bold'), ('color', 'blue')]}]
})

# Форматируем столбец 'Описание' для правильного отображения HTML
styled_df.format({
    'Описание': lambda x: x.replace(" ", "<br />")
})

# Отображаем стилизованный DataFrame
styled_df